A Beginner's Guide to Reinforcement Learning on Grid World with Q-Learning

In [21]:
%reset -f

## Environment

In [22]:
from typing import Literal, get_args, get_origin
import numpy as np
import random

# Base Environment class
class Environment:
    def __init__(self, shape, landmarks: list = None):
        self.shape = shape
        self.landmarks = landmarks
        self.reset()

    def reset(self):
        """Defines the entire state space.

        Is a given within an environment static, i.e. it doesn't change from episode to episode, then this given isn't a state variable. Otherwise, is a given within an
        environment dynamic, i.e. it changes from episode to episode, then this given has to be modeled as a state variable and is part of the entire state space. A dynamic given
        can be observed via its uncertainty.
        """
        self.state = None
        return self.state
    
    def step(self, action):
        """Defines the next state and reward regarding to an action at current state."""
        reward, done = None, False
        return self.state, reward, done

In [23]:
# Literal for actions within the grid world
_MOVETYPES = Literal["up", "down", "left", "right"]

# GridEnvironment class
class GridEnvironment(Environment):
    def __init__(self, shape=(4, 4), landmarks=[(0, 3), (1, 1)]): # (0, 3) is the goal, (1, 1) is a wall
        super().__init__(shape, landmarks)

    def reset(self):
        self.state = (3, 0) # The state of the grid world environment is only defined by the position within the grid
        return self.state

    def give_reward(self):
        if self.state == self.landmarks[0]:
            return 1 # reward for reaching the goal
        elif self.state == self.landmarks[1]:
            return -1 # penalty for hitting the wall
        else:
            return 0 # default reward
    
    def is_terminal(self):
        return self.state == self.landmarks[0]

    def step(self, action: _MOVETYPES):
        # action leads to the transition of the environments state to the next state
        state = list(self.state)
        match(action):
            case "up":
                state[0] = max(0, self.state[0] - 1)
            case "down":
                state[0] = min(self.shape[0] - 1, self.state[0] + 1)
            case "left":
                state[1] = max(0, self.state[1] - 1)
            case "right":
                state[1] = min(self.shape[1] - 1, self.state[1] + 1)
        self.state = tuple(state)

        # check reward or penalty due to the next state
        reward = self.give_reward()

        # check reaching the goal
        done = self.is_terminal()

        return self.state, reward, done

## Agent

In [24]:
# Base Agent class
class Agent:
    def __init__(self, environment: Environment, actions: list = []):
        self.environment = environment
        self.actions = list(actions)
        self.action_space = len(self.actions)

## Q-learning Agent

The Q-learning algorithm uses the following formula to update the Q-value for a state-action-pair:

$Q(s,a) = Q(s,a) + \alpha \cdot [R + \gamma \cdot \underset{a} \max\ Q(s',a') - Q(s,a)]$

with\
$Q(s,a)$ : Q-value for state $s$ and action $a$\
$\alpha$ : learning rate, a value between 0 and 1 that determines how much the old Q-value is updated\
$R$ : immediate reward for taking action $a$ in state $s$\
$\gamma$ : discount factor, a value between 0 and 1 that represents the importance of future rewards. A higher value means that the agent is more focused on long-term rewards\
$\max Q(s',a')$ : maximum Q-value for all possible actions $a'$ in the next state $s'$, representing the best possible reward achievable from that state.

In [ ]:
# QLearningAgent class
class QLearningAgent(Agent):
    """Inherited agent class with Q-Learning algorithm"""

    def __init__(self, environment: Environment, actions: list = [], alpha=0.1, gamma=0.9, epsilon=0.1):
        super().__init__(environment, actions)

        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

        # Q-values for each state-action pair
        # A Q-value, denoted with Q(s,a), represents the expected cumulative reward for choosing action a in state s
        self.q_table = np.zeros(self.environment.shape + (self.action_space,)) # shape of q_table (4, 4, 4)

    # Get best action from a state
    def best_action(self, state):
        return self.actions[np.argmax(self.q_table[state])]
    
    # The agent uses an epsilon-greedy strategy, meaning it randomly explores with probability epsilon and exploits the current knowledge otherwise.
    def choose_action(self, state):
        if random.uniform(0, 1) < self.epsilon:
            return self.actions[np.random.randint(self.action_space)] # Explore
        else:
            return self.best_action(state) # Exploit
    
    # Q-learning update formula to adjust the Q-values after each step.
    def update_q_value(self, state, action, reward, next_state):
        a = self.actions.index(action)
        best_next_q = np.max(self.q_table[next_state]) # Best Q-value for next position
        # Q-learning formula
        self.q_table[state][a] += self.alpha * (reward + self.gamma * best_next_q - self.q_table[state][a])

    # Train agent to get entire Q-table
    def fit(self, episodes=1000, verbose=False):
        for episode in range(episodes):
            state = self.environment.reset() # Reset the environment at the start of each episode
            done = False
            total_reward = 0

            while not done:
                action = self.choose_action(state) # Choose an action
                next_state, reward, done = self.environment.step(action) # Take the action and observe next state, reward
                self.update_q_value(state, action, reward, next_state) # Update Q-values
                state = next_state # Move to the next state
                total_reward += reward
        
            if verbose:
                print(f"Episode: {episode}, Total reward: {total_reward}")

    # Simulate an optimal path to the goal
    def path(self, state=None):
        if state is None:
            state = self.environment.reset()
        else:
            self.environment.state = state
        states = [state]
        actions = []
        done = False

        while not done:
            action = self.best_action(state)
            state, reward, done = self.environment.step(action)
            states.append(state)
            actions.append(action)
            if done:
                break
        
        return states, actions

## Train the Agent

In [33]:
env = GridEnvironment()
agent = QLearningAgent(environment=env, actions=get_args(_MOVETYPES))
agent.fit()

## Use the Trained Agent for Optimal Path

In [34]:
# Example: get best action from a state (3, 0)
state = (3, 0)
best_action = agent.best_action(state)
print(f"Best action from state {state} is: {best_action}")

Best action from state (3, 0) is: up


In [35]:
states, actions = agent.path(state=(2, 1))
print("Optimal path to goal:", *states)
print("Optimal moves from state to goal:", *actions)

Optimal path to goal: (2, 1) (2, 0) (1, 0) (0, 0) (0, 1) (0, 2) (0, 3)
Optimal moves from state to goal: left up up right right right


## Deep-Q-Network Agent

In [ ]:
import numpy as np
import tensorflow as tf
from collections import deque
import random

class DQNAgent(Agent):
    def __init__(self, environment: Environment, state_units, actions: list = [], learning_rate=0.1, gamma=0.99, epsilon=1.0, epsilon_decay=0.995, min_epsilon=0.1, batch_size=32):
        super().__init__(environment, actions)
        
        self.state_units = state_units
        self.memory = deque(maxlen=2000)

        self.learning_rate = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon
        self.batch_size = batch_size

        self.model = self.build_model()

    def build_model(self):
        input = tf.keras.layers.Input(shape=(self.state_units,))
        hidden = tf.keras.layers.Dense(24, activation='relu')(input)
        hidden = tf.keras.layers.Dense(24, activation='relu')(hidden)
        output = tf.keras.layers.Dense(self.action_space, activation='linear')(hidden)
        model = tf.keras.models.Model(inputs=input, outputs=output)
        model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=self.learning_rate))
        return model
    
    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def best_action(self, state):
        q_values = self.model.predict(self.to_tensor(state), verbose=0)
        return self.actions[np.argmax(q_values[0])]

    def choose_action(self, state):
        if np.random.rand() < self.epsilon:
            return self.actions[np.random.randint(self.action_space)]
        else:
            return self.best_action(state)
    
    def to_tensor(self, state):
        return np.asarray(state)[np.newaxis]

    def replay(self):
        if len(self.memory) < self.batch_size:
            return

        minibatch = random.sample(self.memory, self.batch_size)
        states, targets = [], []

        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                next_q_values = self.model.predict(self.to_tensor(next_state), verbose=0)[0]
                target += self.gamma * np.amax(next_q_values)
            q_values = self.model.predict(self.to_tensor(state), verbose=0)[0]
            q_values[self.actions.index(action)] = target

            states.append(state)
            targets.append(q_values)

        self.model.fit(np.array(states), np.array(targets), epochs=1, verbose=0)

        if self.epsilon > self.min_epsilon:
            self.epsilon *= self.epsilon_decay

    def fit(self, episodes=100, verbose=False):
        for episode in range(episodes):
            state = self.environment.reset()
            done = False
            while not done:
                action = self.choose_action(state)
                next_state, reward, done = self.environment.step(action)
                self.remember(state, action, reward, next_state, done)
                state = next_state
            self.replay()

            if verbose:
                print(f"Episode: {episode}, Epsilon: {self.epsilon}")

## Train DQN Agent

In [ ]:
env = GridEnvironment()
agent = DQNAgent(env, state_units=2, actions=get_args(_MOVETYPES))
agent.fit(verbose=True)